# AI-Driven Disease Outbreak Prediction Project Submission Guide

This notebook demonstrates how to collect and organize the required submission items for the group project:

1. Softcopy of manuscript
2. Jupyter file 
3. Datasets

Run the cells below to prepare your submission package.

## 1. Collect Project Files

In this section, we will identify and list the required files:
- Manuscript: README.md (project documentation)
- Jupyter file: This notebook (submission_guide.ipynb)
- Datasets: Files in the data/raw/ directory

In [ ]:
import os
import shutil

# Define file paths
manuscript_file = 'manuscript.md'
jupyter_file = 'submission_guide.ipynb'
dataset_dir = 'data/raw/'

# Collect manuscript
if os.path.exists(manuscript_file):
    print(f"Manuscript found: {manuscript_file}")
else:
    print(f"Manuscript not found: {manuscript_file}")

# Collect Jupyter file (this notebook)
if os.path.exists(jupyter_file):
    print(f"Jupyter file found: {jupyter_file}")
else:
    print(f"Jupyter file not found: {jupyter_file}")

# Collect datasets
if os.path.exists(dataset_dir):
    datasets = [f for f in os.listdir(dataset_dir) if f.endswith('.csv')]
    print(f"Datasets found: {datasets}")
else:
    print(f"Dataset directory not found: {dataset_dir}")

# Store file lists
files_to_submit = {
    'manuscript': manuscript_file,
    'jupyter': jupyter_file,
    'datasets': [os.path.join(dataset_dir, f) for f in datasets]
}

print("Files collected:", files_to_submit)

Manuscript found: manuscript.md
Jupyter file found: submission_guide.ipynb
Datasets found: ['ghana_outbreak_raw.csv']
Files collected: {'manuscript': 'manuscript.md', 'jupyter': 'submission_guide.ipynb', 'datasets': ['data/raw/ghana_outbreak_raw.csv']}


## 2. Organize Files into Folders

Create a submission directory structure and copy the files into appropriate folders.

In [ ]:
# Create submission directory
submission_dir = 'submission_package'
os.makedirs(submission_dir, exist_ok=True)

# Create subdirectories
manuscript_dir = os.path.join(submission_dir, 'manuscript')
jupyter_dir = os.path.join(submission_dir, 'jupyter')
datasets_dir = os.path.join(submission_dir, 'datasets')

os.makedirs(manuscript_dir, exist_ok=True)
os.makedirs(jupyter_dir, exist_ok=True)
os.makedirs(datasets_dir, exist_ok=True)

# Copy files
shutil.copy(files_to_submit['manuscript'], manuscript_dir)
shutil.copy(files_to_submit['jupyter'], jupyter_dir)

for dataset in files_to_submit['datasets']:
    shutil.copy(dataset, datasets_dir)

print("Files organized into submission package.")

Files organized into submission package.


## 3. Create a Submission Archive

Compress the organized files into a ZIP archive for easy submission.

In [ ]:
import zipfile

# Create ZIP archive
archive_name = 'group_project_submission.zip'
with zipfile.ZipFile(archive_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(submission_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, submission_dir)
            zipf.write(file_path, arcname)

print(f"Submission archive created: {archive_name}")

Submission archive created: group_project_submission.zip


## 4. Verify Submission Contents

Check the contents of the archive to ensure all required items are included.

In [ ]:
# Verify archive contents
with zipfile.ZipFile(archive_name, 'r') as zipf:
    file_list = zipf.namelist()
    print("Archive contents:")
    for file in file_list:
        print(f"  - {file}")

# Check for required items
required_items = ['manuscript/manuscript.md', 'jupyter/submission_guide.ipynb', 'datasets/ghana_outbreak_raw.csv']
missing_items = [item for item in required_items if item not in file_list]

if not missing_items:
    print("\nAll required items are present in the archive.")
else:
    print(f"\nMissing items: {missing_items}")

print(f"\nSubmission package ready: {archive_name}")

Archive contents:
  - datasets/ghana_outbreak_raw.csv
  - jupyter/submission_guide.ipynb
  - manuscript/manuscript.md

All required items are present in the archive.

Submission package ready: group_project_submission.zip


## 5. Development, Testing, and Model Training

This section demonstrates practical model development and testing steps for the AI outbreak prediction project. It includes data loading, preprocessing, model training (RandomForest + XGBoost), evaluation, and quick API sanity checks.

In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Load dataset
raw_path = 'data/raw/ghana_outbreak_raw.csv'
if not os.path.exists(raw_path):
    raise FileNotFoundError(f"Dataset not found: {raw_path}")

df = pd.read_csv(raw_path)
print('Dataset shape:', df.shape)
print(df.head())

# Minimal preprocessing
feature_cols = [c for c in df.columns if c not in ['outbreak_label']]
if 'outbreak_label' not in df.columns:
    raise ValueError("Target column 'outbreak_label' missing")

X = df[feature_cols].copy()
y = df['outbreak_label']

# Categorical handling if needed
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category').cat.codes

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

Dataset shape: (1000, 9)
     district        date  rainfall_mm  temperature_c  humidity_pct  \
0  Cape Coast  2023-01-01    13.449943      23.173392     59.165620   
1    Takoradi  2023-01-02     4.163577      25.923402     45.503548   
2      Tamale  2023-01-03     5.964354      18.264369     50.781402   
3    Takoradi  2023-01-04    12.787347      20.141930     49.960426   
4    Takoradi  2023-01-05     6.141987      31.002070     62.072673   

   sanitation_score  population_density  previous_cases  outbreak_label  
0          0.716386          485.516621              23               0  
1          0.797840          124.395987              24               0  
2          0.540308          805.151352              20               0  
3          0.969784          109.461902              18               0  
4          0.877567          785.700905              18               0  
Train shape: (750, 8) Test shape: (250, 8)


### 5.1 Model Training (RandomForest + XGBoost)

Train two models in separate cells to keep the workflow modular and clear.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train RandomForest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print('RF Accuracy:', accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


RF Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       250

    accuracy                           1.00       250
   macro avg       1.00      1.00      1.00       250
weighted avg       1.00      1.00      1.00       250



In [5]:
from xgboost import XGBClassifier

# Train XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print('XGB Accuracy:', accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


XGB Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       250

    accuracy                           1.00       250
   macro avg       1.00      1.00      1.00       250
weighted avg       1.00      1.00      1.00       250



c:\Users\JOSEPH KOJO KUTTOR\Desktop\AI Project\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:13:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### 5.2 Save Models and Test API Integration

Save trained models and optionally verify the API endpoint works correctly.

In [6]:
import joblib

# Save models
os.makedirs('models', exist_ok=True)
joblib.dump(rf, 'models/model_randomforest.joblib')
joblib.dump(xgb, 'models/model_xgboost.joblib')

training_results = {
    'rf_accuracy': accuracy_score(y_test, y_pred_rf),
    'xgb_accuracy': accuracy_score(y_test, y_pred_xgb)
}
print('Training results:', training_results)
print('Model artifacts saved in models/')

# Optional: API health check (requires API server running)
try:
    import requests
    res = requests.get('http://127.0.0.1:5000/status', timeout=5)
    print('API status:', res.json())
except Exception as e:
    print('API status check failed:', e)


Training results: {'rf_accuracy': 1.0, 'xgb_accuracy': 1.0}
Model artifacts saved in models/
API status check failed: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /status (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
